# About the data

- **bureau.csv**: Application data from previous loans that clients obtained from other institutions and that were reported to the credit bureau.  
  *Granularity: one row per client's loan in the Credit Bureau.*  

- **bureau_balance.csv**: The monthly balance of credits in the credit bureau.  
  *This represents the behavioral/temporal data.*  

---

## The Bureau

This dataset provides a **snapshot of the client’s credit situation at the time of their Home Credit application**:

- **Portfolio of external loans**: how many loans the client currently has outside Home Credit.  
- **Credit type**: consumer loans, car loans, credit cards, etc.  
- **Credit status**: Active, Closed, Bad debt, Sold, etc.  
- **Credit size**: original credit amount, current debt, overdue debt, credit card limits.  
- **Loan history**: how many times the loan was prolonged, expected end date, whether the loan is already closed.  
- **Risk indicators**:  
  - `AMT_CREDIT_SUM_OVERDUE` > 0 → client currently overdue.  
  - Large `CREDIT_DAY_OVERDUE` → severe delinquency in the past.  
  - `AMT_CREDIT_MAX_OVERDUE` → maximum overdue amount ever recorded.  

---

## The Bureau Balance

This dataset provides a **time-series history of repayment behavior for each loan**:

- **Monthly credit status**: share of months in status `0` (on time) vs `1–5` (various delinquency buckets).  
- **Severity of delinquency**: whether the loan ever reached DPD > 120, longest consecutive delinquent streak.  
- **Repayment completion**: whether loans transitioned from Active → Closed on schedule.  
- **Stability**: presence of “X” (unknown) months or irregular jumps in status.  
- **Recency**: whether the client was delinquent in the last 6–12 months before the application.  

---

## Combined View (Aggregated to SK_ID_CURR)

By merging both datasets, we can derive:  
- Total number of external loans.  
- Ratio of closed loans vs active loans.  
- Number of loans ever delinquent.  
- Worst delinquency level across all previous loans.  
- Debt-to-Credit ratio (current debt vs total credit).  
- Recency of delinquency (recent overdue behavior).  

Together, **`bureau.csv` offers a snapshot** of the client’s external credit profile, while **`bureau_balance.csv` provides longitudinal behavioral patterns**.  


# Import Libraries

In [ ]:
import polars as pl
import warnings
import os 

warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)
os.environ['POLARS_WARN_UNSTABLE'] = '0'

In [ ]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

# Import Data

In [ ]:
# Load all data files using Polars
print("Loading datasets with Polars...")
# Additional data sources
bureau_df = pl.read_csv('../data/bureau.csv')
bureau_balance_df = pl.read_csv('../data/bureau_balance.csv')
print(f"Bureau data shape: {bureau_df.shape}")
print(f"Bureau balance data shape: {bureau_balance_df.shape}")

# EDA

In [ ]:
# Basic data inspection
print("=== BUREAU.CSV OVERVIEW ===")
print(f"Bureau shape: {bureau_df.shape}")
print(f"Bureau columns: {list(bureau_df.columns)}")
print(f"\nUnique clients in bureau: {bureau_df['SK_ID_CURR'].n_unique()}")
print(f"Unique bureau records: {bureau_df['SK_ID_BUREAU'].n_unique()}")
print(f"Average bureau records per client: {bureau_df.shape[0] / bureau_df['SK_ID_CURR'].n_unique():.2f}")

print("\n=== BUREAU_BALANCE.CSV OVERVIEW ===")
print(f"Bureau balance shape: {bureau_balance_df.shape}")
print(f"Bureau balance columns: {list(bureau_balance_df.columns)}")
print(f"Unique bureau IDs in balance: {bureau_balance_df['SK_ID_BUREAU'].n_unique()}")
print(f"Average balance records per bureau: {bureau_balance_df.shape[0] / bureau_balance_df['SK_ID_BUREAU'].n_unique():.2f}")

# Check if all bureau IDs in balance exist in bureau
bureau_ids_in_bureau = set(bureau_df['SK_ID_BUREAU'].to_list())
bureau_ids_in_balance = set(bureau_balance_df['SK_ID_BUREAU'].to_list())
print(f"\nBureau IDs overlap: {len(bureau_ids_in_balance.intersection(bureau_ids_in_bureau))} / {len(bureau_ids_in_balance)} balance records have matching bureau records")

In [ ]:
# Detailed EDA for Bureau data
print("=== BUREAU.CSV DETAILED ANALYSIS ===")

# Data types and missing values
print("\nData types and missing values:")
print(bureau_df.schema)

# Missing value analysis
missing_analysis = []
for col in bureau_df.columns:
    null_count = bureau_df[col].null_count()
    null_pct = (null_count / bureau_df.shape[0]) * 100
    missing_analysis.append({'column': col, 'null_count': null_count, 'null_percentage': null_pct})

missing_df = pl.DataFrame(missing_analysis).sort('null_percentage', descending=True)
print(f"\nMissing values analysis:")
print(missing_df)

# Look at first few rows
print(f"\nFirst 5 rows of bureau data:")
print(bureau_df.head())

# Unique values for categorical columns
categorical_cols = ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']
print(f"\nUnique values in categorical columns:")
for col in categorical_cols:
    unique_vals = bureau_df[col].value_counts().sort('count', descending=True)
    print(f"\n{col}:")
    print(unique_vals)

### About Data Quality and Missing Values
- The total number of consumer credit is the majority, the currency is almost one type (may indicate as less predictive power) and bad dept is very rare. 
- About the data quality and missing values:
    - AMT_ANNUITY: almost every loan from credit bureau does not have the annuity payment, this could be the type of product since some of them may not have a clear annuity. **Could be a sign that we must remove these features**
    - AMT_CREDIT_MAX_OVERDUE: about 65%, this maybe because most of the customer loan is never late or credit bureau dont save the information. **This could be just a Yes/No indicator**. 
    - DAYS_ENDATE_FACT: This onnly have value when the contract is closed, while remain null when the contract is active. 
    - AMT_CREDIT_SUM_LIMIT: only apply for credit cards, we miss a lot because the majoirtiy is consumer credit. 
    - AMT_CREDIT_SUM_DEBT: this could be null when the loan is already pay.

=> We can see that most of the missing value is due to logic, not because of data quality issue. We can overcome these missing value by creating flags such as HAS_CREDIT_LIMIT, HAS_ENDDATE_FACT

---

### About Credit Active
- We have 4 main categories:
    - Closed: Most of them, meaning that most of the customer have done their payment before send a new loan application for the company. 
    - Active: Quite a few, some of our customer still not fulfill there payment with outside comapny yet but they still apply for our anyway. 
    - Sold: Some of the debt of our customer have been sold for 3rd party (kinda risky)
    - Bad debt: 21 cases, very risky, we could eliminate those with bad debt as the preprocessing/postprocessing. 

--- 

### About Credit Currency
- Most of them focus on Currency 1, therefore not many things we can do with this. Just ignore the column.

--- 

### About Credit Type. 
- The distribution of this is very skew:
    - Consumer Credit took the majority, this could be a sign for us to be careful when performing some of the categorical encoding because this may cause havoc, we should try and do the frequency encoding for this. 

In [ ]:
# Numerical columns analysis for Bureau
print("=== NUMERICAL COLUMNS STATISTICS ===")

numerical_cols = [col for col in bureau_df.columns if col not in ['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE']]
print(f"Numerical columns: {numerical_cols}")

# Statistical summary
numeric_stats = bureau_df.select(numerical_cols).describe()
print(f"\nStatistical Summary:")
print(numeric_stats)

# Distribution analysis for key amount columns
amount_cols = [col for col in numerical_cols if 'AMT_' in col]
bureau_df = bureau_df.with_columns(pl.col('AMT_ANNUITY').cast(pl.Float64))  
print(f"\nAmount columns analysis:")
for col in amount_cols:
    col_stats = bureau_df.select([
        pl.col(col).min().alias('min'),
        pl.col(col).max().alias('max'), 
        pl.col(col).mean().alias('mean'),
        pl.col(col).median().alias('median'),
        pl.col(col).std().alias('std'),
        (pl.col(col) == 0).sum().alias('zero_count'),
        pl.col(col).null_count().alias('null_count')
    ])
    print(f"\n{col}:")
    print(col_stats)

# Days columns analysis (negative values expected as they're relative to application)
days_cols = [col for col in numerical_cols if 'DAYS_' in col]
print(f"\nDays columns analysis (negative = days before application):")
for col in days_cols:
    col_stats = bureau_df.select([
        pl.col(col).min().alias('min'),
        pl.col(col).max().alias('max'),
        pl.col(col).mean().alias('mean'),
        pl.col(col).median().alias('median'),
        (pl.col(col) < 0).sum().alias('negative_count'),
        (pl.col(col) > 0).sum().alias('positive_count'),
        pl.col(col).null_count().alias('null_count')
    ])
    print(f"\n{col}:")
    print(col_stats)

In [ ]:
# Bureau Balance EDA
print("=== BUREAU_BALANCE.CSV DETAILED ANALYSIS ===")

print(f"\nBureau balance shape: {bureau_balance_df.shape}")
print(f"Bureau balance columns: {list(bureau_balance_df.columns)}")


# Missing value analysis for bureau_balance
print(f"\nMissing values in bureau_balance:")
for col in bureau_balance_df.columns:
    null_count = bureau_balance_df[col].null_count()
    null_pct = (null_count / bureau_balance_df.shape[0]) * 100
    print(f"{col}: {null_count} ({null_pct:.2f}%)")

# Look at first few rows
print(f"\nFirst 10 rows of bureau_balance:")
print(bureau_balance_df.head(10))

# STATUS column analysis (this is the key behavioral indicator)
print(f"\nSTATUS column analysis:")
status_counts = bureau_balance_df['STATUS'].value_counts().sort('count', descending=True)
print(status_counts)

# MONTHS_BALANCE analysis (timeline of records)
print(f"\nMONTHS_BALANCE analysis:")
months_stats = bureau_balance_df.select([
    pl.col('MONTHS_BALANCE').min().alias('min_month'),
    pl.col('MONTHS_BALANCE').max().alias('max_month'),
    pl.col('MONTHS_BALANCE').mean().alias('mean_month'),
    pl.col('MONTHS_BALANCE').median().alias('median_month'),
    pl.col('MONTHS_BALANCE').n_unique().alias('unique_months')
])
print(months_stats)

months_dist = bureau_balance_df['MONTHS_BALANCE'].value_counts().sort('MONTHS_BALANCE')
print(f"\nMonths balance distribution (showing first 20):")
print(months_dist.head(20))

Here are some insights from the bureau_balance.csv

### About data
- Clean, no missing value
- Granularity is around 27Mil with 
- Most of them are closed contract or haven't got to the DBD yet. 

In [ ]:
# Data Quality and Relationship Analysis
print("=== DATA QUALITY AND RELATIONSHIPS ===")

# 1. Analyze client distribution - how many bureau records per client?
client_bureau_counts = bureau_df.group_by('SK_ID_CURR').agg([
    pl.count().alias('bureau_records_count'),
    pl.col('CREDIT_ACTIVE').n_unique().alias('unique_statuses'),
    pl.col('CREDIT_TYPE').n_unique().alias('unique_credit_types')
])

print("Client bureau records distribution:")
bureau_count_dist = client_bureau_counts['bureau_records_count'].value_counts().sort('bureau_records_count')
print(bureau_count_dist.head(20))

print(f"\nClients with multiple bureau records: {(client_bureau_counts['bureau_records_count'] > 1).sum()}")
print(f"Clients with single bureau record: {(client_bureau_counts['bureau_records_count'] == 1).sum()}")
print(f"Max bureau records per client: {client_bureau_counts['bureau_records_count'].max()}")

# 2. Credit status combinations per client
print(f"\n=== CREDIT STATUS PATTERNS PER CLIENT ===")
status_patterns = client_bureau_counts.group_by(['unique_statuses', 'unique_credit_types']).agg([
    pl.count().alias('client_count')
]).sort('client_count', descending=True)
print("Top status/credit type combinations:")
print(status_patterns.head(10))

# 3. Active vs Closed credit analysis
print(f"\n=== ACTIVE VS CLOSED CREDIT ANALYSIS ===")
active_closed_analysis = bureau_df.group_by('SK_ID_CURR').agg([
    (pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('active_count'),
    (pl.col('CREDIT_ACTIVE') == 'Closed').sum().alias('closed_count'),
    (pl.col('CREDIT_ACTIVE') == 'Bad debt').sum().alias('bad_debt_count'),
    pl.count().alias('total_count')
])

print("Active/Closed credit distribution per client:")
print("Active credits per client:")
print(active_closed_analysis['active_count'].value_counts().sort('active_count').head(10))

print("\nClosed credits per client:")  
print(active_closed_analysis['closed_count'].value_counts().sort('closed_count').head(10))

print(f"\nClients with bad debt: {(active_closed_analysis['bad_debt_count'] > 0).sum()}")
print(f"Total bad debt records: {active_closed_analysis['bad_debt_count'].sum()}")

Some insight on this bureau data
- Lots of customer only have from little to maybe almost 10 credit request outside the company. Currently we are still struggle to see how this will match up with our feature engineering process, one thing may come in mind is that we  create a normalized feature that balance between the number of loaning outside the company with the status closed, let say when a customer have lots of outside credit application outside the company and most of the time they can manage to fulfill the loan on time, then that person is good

In [ ]:
# FEATURE ENGINEERING: Risk Indicators from Bureau Data
print("=== BUREAU RISK INDICATORS FEATURE ENGINEERING ===")

# Create comprehensive risk features at SK_ID_CURR level
bureau_risk_features = bureau_df.group_by('SK_ID_CURR').agg([
    # Basic counts and diversity
    pl.count().alias('BUREAU_TOTAL_COUNT'),
    pl.col('CREDIT_TYPE').n_unique().alias('BUREAU_CREDIT_TYPES_COUNT'),
    
    # Credit status risk indicators
    (pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_COUNT'),
    (pl.col('CREDIT_ACTIVE') == 'Closed').sum().alias('BUREAU_CLOSED_COUNT'),
    (pl.col('CREDIT_ACTIVE') == 'Bad debt').sum().alias('BUREAU_BAD_DEBT_COUNT'),
    (pl.col('CREDIT_ACTIVE') == 'Sold').sum().alias('BUREAU_SOLD_COUNT'),
    
    # High-risk status ratios
    ((pl.col('CREDIT_ACTIVE') == 'Bad debt').sum() / pl.count()).alias('BUREAU_BAD_DEBT_RATIO'),
    ((pl.col('CREDIT_ACTIVE') == 'Sold').sum() / pl.count()).alias('BUREAU_SOLD_RATIO'),
    (((pl.col('CREDIT_ACTIVE') == 'Bad debt') | (pl.col('CREDIT_ACTIVE') == 'Sold')).sum() / pl.count()).alias('BUREAU_HIGH_RISK_RATIO'),
    
    # Overdue risk indicators
    pl.col('CREDIT_DAY_OVERDUE').sum().alias('BUREAU_OVERDUE_DAYS_TOTAL'),
    pl.col('CREDIT_DAY_OVERDUE').mean().alias('BUREAU_OVERDUE_DAYS_MEAN'),
    pl.col('CREDIT_DAY_OVERDUE').max().alias('BUREAU_OVERDUE_DAYS_MAX'),
    (pl.col('CREDIT_DAY_OVERDUE') > 0).sum().alias('BUREAU_OVERDUE_COUNT'),
    ((pl.col('CREDIT_DAY_OVERDUE') > 0).sum() / pl.count()).alias('BUREAU_OVERDUE_RATIO'),
    
    # Amount-based overdue risk
    pl.col('AMT_CREDIT_SUM_OVERDUE').sum().alias('BUREAU_AMT_OVERDUE_TOTAL'),
    pl.col('AMT_CREDIT_SUM_OVERDUE').mean().alias('BUREAU_AMT_OVERDUE_MEAN'),
    pl.col('AMT_CREDIT_SUM_OVERDUE').max().alias('BUREAU_AMT_OVERDUE_MAX'),
    pl.col('AMT_CREDIT_MAX_OVERDUE').max().alias('BUREAU_AMT_MAX_OVERDUE_EVER'),
    (pl.col('AMT_CREDIT_SUM_OVERDUE') > 0).sum().alias('BUREAU_AMT_OVERDUE_COUNT'),
    ((pl.col('AMT_CREDIT_SUM_OVERDUE') > 0).sum() / pl.count()).alias('BUREAU_AMT_OVERDUE_RATIO'),
    
    # Credit prolongation risk (refinancing frequency)
    pl.col('CNT_CREDIT_PROLONG').sum().alias('BUREAU_PROLONG_TOTAL'),
    pl.col('CNT_CREDIT_PROLONG').mean().alias('BUREAU_PROLONG_MEAN'),
    pl.col('CNT_CREDIT_PROLONG').max().alias('BUREAU_PROLONG_MAX'),
    (pl.col('CNT_CREDIT_PROLONG') > 0).sum().alias('BUREAU_PROLONG_COUNT'),
    ((pl.col('CNT_CREDIT_PROLONG') > 0).sum() / pl.count()).alias('BUREAU_PROLONG_RATIO'),
])

print(f"Bureau risk features shape: {bureau_risk_features.shape}")
print(f"Risk features columns: {bureau_risk_features.columns}")
print(f"\nFirst 5 rows of risk features:")
print(bureau_risk_features.head())

In [ ]:
# ADVANCED RISK FEATURES: Credit Utilization and Financial Stress
print("=== ADVANCED BUREAU RISK FEATURES ===")

bureau_advanced_risk = bureau_df.group_by('SK_ID_CURR').agg([
    # Credit utilization for credit cards (debt vs limit)
    (pl.col('AMT_CREDIT_SUM_DEBT').sum() / pl.col('AMT_CREDIT_SUM_LIMIT').sum()).alias('BUREAU_CREDIT_UTILIZATION_RATIO'),
    
    # Financial stress indicators
    (pl.col('AMT_CREDIT_SUM_DEBT').sum() / pl.col('AMT_CREDIT_SUM').sum()).alias('BUREAU_DEBT_TO_CREDIT_RATIO'),
    (pl.col('AMT_CREDIT_SUM_OVERDUE').sum() / pl.col('AMT_CREDIT_SUM').sum()).alias('BUREAU_OVERDUE_TO_CREDIT_RATIO'),
    
    # Active credit portfolio risk
    pl.col('AMT_CREDIT_SUM').filter(pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_CREDIT_SUM'),
    pl.col('AMT_CREDIT_SUM_DEBT').filter(pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_DEBT_SUM'),
    pl.col('AMT_CREDIT_SUM_OVERDUE').filter(pl.col('CREDIT_ACTIVE') == 'Active').sum().alias('BUREAU_ACTIVE_OVERDUE_SUM'),
    
    # Credit limit utilization for active credit cards only
    (pl.col('AMT_CREDIT_SUM_DEBT').filter(pl.col('CREDIT_ACTIVE') == 'Active').sum() / 
     pl.col('AMT_CREDIT_SUM_LIMIT').filter(pl.col('CREDIT_ACTIVE') == 'Active').sum()).alias('BUREAU_ACTIVE_UTILIZATION_RATIO'),
    
    # Maxed out credit indicators
    (pl.col('AMT_CREDIT_SUM_DEBT') >= pl.col('AMT_CREDIT_SUM_LIMIT')).sum().alias('BUREAU_MAXED_OUT_COUNT'),
    ((pl.col('AMT_CREDIT_SUM_DEBT') >= pl.col('AMT_CREDIT_SUM_LIMIT')).sum() / pl.count()).alias('BUREAU_MAXED_OUT_RATIO'),
    
    # High utilization indicators (>80% of limit)
    (pl.col('AMT_CREDIT_SUM_DEBT') > pl.col('AMT_CREDIT_SUM_LIMIT') * 0.8).sum().alias('BUREAU_HIGH_UTIL_COUNT'),
    ((pl.col('AMT_CREDIT_SUM_DEBT') > pl.col('AMT_CREDIT_SUM_LIMIT') * 0.8).sum() / pl.count()).alias('BUREAU_HIGH_UTIL_RATIO'),
])

print(f"Advanced bureau risk features shape: {bureau_advanced_risk.shape}")
print(f"Advanced features columns: {bureau_advanced_risk.columns}")
print(f"\nAdvanced risk features sample:")
print(bureau_advanced_risk.head())

In [ ]:
# BUREAU BALANCE RISK FEATURES: Payment Behavior Patterns  
print("=== BUREAU BALANCE BEHAVIORAL RISK FEATURES ===")

# Create behavioral features from bureau_balance
bureau_balance_risk = bureau_balance_df.group_by('SK_ID_BUREAU').agg([
    # Total months observed
    pl.count().alias('TOTAL_MONTHS_OBSERVED'),
    
    # Payment performance indicators
    (pl.col('STATUS') == '0').sum().alias('MONTHS_ON_TIME'),
    (pl.col('STATUS') == 'C').sum().alias('MONTHS_CLOSED'),
    (pl.col('STATUS') == 'X').sum().alias('MONTHS_UNKNOWN'),
    
    # DPD severity breakdown
    (pl.col('STATUS') == '1').sum().alias('MONTHS_DPD_1_30'),
    (pl.col('STATUS') == '2').sum().alias('MONTHS_DPD_31_60'), 
    (pl.col('STATUS') == '3').sum().alias('MONTHS_DPD_61_90'),
    (pl.col('STATUS') == '4').sum().alias('MONTHS_DPD_91_120'),
    (pl.col('STATUS') == '5').sum().alias('MONTHS_DPD_120_PLUS'),
    
    # Timeline features
    pl.col('MONTHS_BALANCE').min().alias('EARLIEST_MONTH'),
    pl.col('MONTHS_BALANCE').max().alias('LATEST_MONTH'),
])

# Calculate derived risk ratios
bureau_balance_risk = bureau_balance_risk.with_columns([
    # Total DPD months and ratios
    (pl.col('MONTHS_DPD_1_30') + pl.col('MONTHS_DPD_31_60') + 
     pl.col('MONTHS_DPD_61_90') + pl.col('MONTHS_DPD_91_120') + 
     pl.col('MONTHS_DPD_120_PLUS')).alias('TOTAL_DPD_MONTHS'),
    
    # Performance ratios
    (pl.col('MONTHS_ON_TIME') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('ON_TIME_RATIO'),
    (pl.col('MONTHS_UNKNOWN') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('UNKNOWN_RATIO'),
    
    # Risk flags
    (pl.col('MONTHS_DPD_120_PLUS') > 0).alias('HAS_SEVERE_DPD'),
    (pl.col('MONTHS_DPD_1_30') + pl.col('MONTHS_DPD_31_60') + 
     pl.col('MONTHS_DPD_61_90') + pl.col('MONTHS_DPD_91_120') + 
     pl.col('MONTHS_DPD_120_PLUS') > 0).alias('HAS_ANY_DPD'),
])

# Add DPD ratios
bureau_balance_risk = bureau_balance_risk.with_columns([
    ((pl.col('MONTHS_DPD_1_30') + pl.col('MONTHS_DPD_31_60') + 
      pl.col('MONTHS_DPD_61_90') + pl.col('MONTHS_DPD_91_120') + 
      pl.col('MONTHS_DPD_120_PLUS')) / pl.col('TOTAL_MONTHS_OBSERVED')).alias('DPD_RATIO'),
    
    (pl.col('MONTHS_DPD_120_PLUS') / pl.col('TOTAL_MONTHS_OBSERVED')).alias('SEVERE_DPD_RATIO')
])

print(f"Bureau balance risk features shape: {bureau_balance_risk.shape}")
print(f"Bureau balance risk columns: {bureau_balance_risk.columns}")
print(f"\nBureau balance risk features sample:")
print(bureau_balance_risk.head())

In [ ]:
# COMBINE ALL BUREAU FEATURES AT CLIENT LEVEL (SK_ID_CURR)
print("=== COMBINING ALL BUREAU FEATURES ===")

# First, join bureau_balance_risk back to bureau to get SK_ID_CURR mapping
bureau_with_balance_risk = bureau_df.select(['SK_ID_CURR', 'SK_ID_BUREAU']).join(
    bureau_balance_risk, 
    on='SK_ID_BUREAU', 
    how='left'
)

# Aggregate bureau_balance features to SK_ID_CURR level
bureau_balance_client_risk = bureau_with_balance_risk.group_by('SK_ID_CURR').agg([
    # Count of bureau records with balance history
    pl.col('SK_ID_BUREAU').count().alias('BUREAU_WITH_BALANCE_COUNT'),
    
    # Aggregate behavioral metrics across all client's bureau records
    pl.col('TOTAL_MONTHS_OBSERVED').sum().alias('TOTAL_MONTHS_ALL_BUREAUS'),
    pl.col('MONTHS_ON_TIME').sum().alias('TOTAL_MONTHS_ON_TIME'),
    pl.col('TOTAL_DPD_MONTHS').sum().alias('TOTAL_DPD_ALL_BUREAUS'),
    pl.col('MONTHS_DPD_120_PLUS').sum().alias('TOTAL_SEVERE_DPD_MONTHS'),
    
    # Worst behavior indicators
    pl.col('DPD_RATIO').max().alias('WORST_DPD_RATIO'),
    pl.col('SEVERE_DPD_RATIO').max().alias('WORST_SEVERE_DPD_RATIO'),
    pl.col('ON_TIME_RATIO').min().alias('WORST_ON_TIME_RATIO'),
    
    # Average behavior across all bureau records
    pl.col('DPD_RATIO').mean().alias('AVG_DPD_RATIO'),
    pl.col('ON_TIME_RATIO').mean().alias('AVG_ON_TIME_RATIO'),
    
    # Risk flags
    pl.col('HAS_SEVERE_DPD').sum().alias('COUNT_BUREAUS_WITH_SEVERE_DPD'),
    pl.col('HAS_ANY_DPD').sum().alias('COUNT_BUREAUS_WITH_ANY_DPD'),
])

# Calculate final client-level ratios
bureau_balance_client_risk = bureau_balance_client_risk.with_columns([
    # Overall payment performance across all bureau records
    (pl.col('TOTAL_MONTHS_ON_TIME') / pl.col('TOTAL_MONTHS_ALL_BUREAUS')).alias('OVERALL_ON_TIME_RATIO'),
    (pl.col('TOTAL_DPD_ALL_BUREAUS') / pl.col('TOTAL_MONTHS_ALL_BUREAUS')).alias('OVERALL_DPD_RATIO'),
    (pl.col('TOTAL_SEVERE_DPD_MONTHS') / pl.col('TOTAL_MONTHS_ALL_BUREAUS')).alias('OVERALL_SEVERE_DPD_RATIO'),
    
    # Risk flags at client level
    (pl.col('COUNT_BUREAUS_WITH_SEVERE_DPD') > 0).alias('CLIENT_HAS_SEVERE_DPD_HISTORY'),
    (pl.col('COUNT_BUREAUS_WITH_ANY_DPD') > 0).alias('CLIENT_HAS_ANY_DPD_HISTORY'),
])

print(f"Bureau balance client features shape: {bureau_balance_client_risk.shape}")

# Combine all bureau risk features
final_bureau_features = bureau_risk_features.join(
    bureau_advanced_risk, 
    on='SK_ID_CURR',
    how='left'
).join(
    bureau_balance_client_risk,
    on='SK_ID_CURR', 
    how='left'
)

print(f"Final combined bureau features shape: {final_bureau_features.shape}")
print(f"Total features: {len(final_bureau_features.columns) - 1}")  # -1 for SK_ID_CURR
print(f"Final feature columns:")
for i, col in enumerate(final_bureau_features.columns[1:], 1):  # Skip SK_ID_CURR
    print(f"{i:2d}. {col}")

print(f"\nSample of final features:")
print(final_bureau_features.head())

In [ ]:
# SAVE BUREAU FEATURES AND PREPARE FOR JOINING
print("=== SAVING BUREAU FEATURES ===")

# Save to CSV for later use
output_path = '../data/bureau_risk_features.csv'
final_bureau_features.write_csv(output_path)
print(f"Bureau risk features saved to: {output_path}")

# Show summary statistics
print(f"\n=== BUREAU FEATURES SUMMARY ===")
print(f"Total clients with bureau data: {final_bureau_features.shape[0]:,}")
print(f"Total risk features created: {len(final_bureau_features.columns) - 1}")

# Check for missing values in key risk features
key_risk_features = [
    'BUREAU_HIGH_RISK_RATIO', 
    'BUREAU_OVERDUE_RATIO', 
    'BUREAU_DEBT_TO_CREDIT_RATIO',
    'OVERALL_DPD_RATIO',
    'CLIENT_HAS_SEVERE_DPD_HISTORY'
]

print(f"\nMissing values in key risk features:")
for feature in key_risk_features:
    if feature in final_bureau_features.columns:
        missing_count = final_bureau_features[feature].null_count()
        missing_pct = (missing_count / final_bureau_features.shape[0]) * 100
        print(f"{feature}: {missing_count} ({missing_pct:.1f}%)")

print(f"\n=== HOW TO JOIN WITH APPLICATION_TRAIN ===")
print("# Load application_train")
print("# application_train = pl.read_csv('../data/application_train.csv')")
print("#")
print("# Join bureau features")
print("# final_dataset = application_train.join(")
print("#     final_bureau_features,")
print("#     on='SK_ID_CURR',")
print("#     how='left'")
print("# )")
print("#")
print("# This will add all bureau risk features to your main modeling dataset!")

print(f"\n=== FEATURE ENGINEERING COMPLETE ===")
print("Bureau risk indicators successfully created at SK_ID_CURR granularity!")
print(f"Ready to join with application_train for model training.")

# Update todo
print(f"\n✅ Bureau feature engineering completed successfully!")